<a href="https://colab.research.google.com/github/ocayaro/Ocaya-Yakuphanoglu-method-program/blob/main/pla_pcl_latest.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import sys
import os

# Install condacolab
!pip install -q condacolab
import condacolab
condacolab.install()

# Install OpenMM, OpenFF, and dependencies via mamba
!mamba install -c conda-forge openmm openmmforcefields openff-toolkit rdkit mdanalysis matplotlib -y
!mamba install -c conda-forge openff-packmol -y

# Ensure Conda site-packages is in Python's search path *before* further imports
# This often needs to happen after condacolab.install() to ensure the correct path is found.
conda_prefix = os.environ.get('CONDA_PREFIX', '/usr/local')
conda_site_packages = os.path.join(conda_prefix, 'lib', f'python{sys.version_info.major}.{sys.version_info.minor}', 'site-packages')
if conda_site_packages not in sys.path:
    sys.path.append(conda_site_packages)
    print(f"Added {conda_site_packages} to sys.path to ensure conda packages are found.")

# Fallback to pip install if mamba install doesn't resolve the ModuleNotFoundError.
# Note: A runtime restart and re-running all cells will still be required after this.
!pip install -q openff-packmol

# Explicitly install rdkit as a fallback if mamba or previous pip fails
!pip install -q rdkit

# Verifying openff-packmol installation
print('Verifying openff-packmol installation:')
!pip list | grep openff-packmol

print('Verifying rdkit installation:')
!pip list | grep rdkit

# Robust check for RDKit import
try:
    import rdkit
    print('\n[SUCCESS] RDKit is successfully installed and importable within this cell.')
except ImportError:
    print('\n[ERROR] RDKit could not be imported after installation attempts.')
    print('This often happens if the Python environment is not fully refreshed.')

print("""\n--- IMPORTANT: PLEASE RESTART THE RUNTIME (Runtime > Restart runtime) AND THEN 'Run all' CELLS! --- Some packages may conflict with pre-installed Colab packages.
Only after a full restart and re-running all cells will the new installations be correctly recognized by all parts of the notebook.\n""")

# Fix for torchvision::nms RuntimeError: Uninstall and reinstall compatible torch/torchvision
print('\n[FIX] Attempting to resolve torchvision conflict by reinstalling torch/torchvision...')
!pip uninstall -y torch torchvision torchaudio
# Install a common compatible version of torch and torchvision for Colab (assuming CUDA 12.1)
# You might need to adjust 'cu121' based on your Colab GPU runtime version
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
print('[FIX] Reinstallation of torch/torchvision completed.')

# Fix for openff.units ImportError: Cannot import name 'get_current_backend'
print('\n[FIX] Attempting to resolve openff.units conflict by reinstalling openff packages...')
!mamba uninstall -y openff-toolkit openff-units openff-packmol
!mamba install -c conda-forge openff-toolkit openff-packmol -y
print('[FIX] Reinstallation of openff packages completed. Please re-run all cells after a runtime restart.')

# Fix for scipy.optimize ImportError: CXXABI_1.3.15' not found
print('\n[FIX] Attempting to resolve scipy CXXABI conflict by reinstalling scipy...')
!mamba uninstall -y scipy numpy # Uninstall scipy and its common dependency numpy
!mamba install -c conda-forge scipy numpy -y # Using mamba for a consistent environment
print('[FIX] Reinstallation of scipy completed. Please re-run all cells after a runtime restart.')

conda_site_packages = "/usr/local/lib/python3.12/site-packages"
if conda_site_packages not in sys.path:
    sys.path.append(conda_site_packages)
from openff.packmol import pack_box

import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit

from rdkit import Chem
from rdkit.Chem import AllChem, Descriptors
from openff.toolkit.topology import Molecule

import openmm as mm
import openmm.app as app
from openmm import unit
from openmmforcefields.generators import GAFFTemplateGenerator

from openff.units import unit as openff_unit
from google.colab import files



# =================================================================================
# USER PARAMETERS
# =================================================================================

GLOBAL_RANDOM_SEED = 42          # fixed seed for reproducible embedding + dynamics
DP_PLA = 20                      # degree of polymerization for PLA chains
DP_PCL = 20                      # degree of polymerization for PCL chains
N_EMBED_ATTEMPTS = 20            # number of conformer-embedding attempts per chain

np.random.seed(GLOBAL_RANDOM_SEED)

# =================================================================================
# DIAGNOSTICS / TOOLCHAIN CHECKS (unchanged from the original script)
# =================================================================================

platforms = [mm.Platform.getPlatform(i).getName() for i in range(mm.Platform.getNumPlatforms())]
print(f"[DIAGNOSTICS] Available OpenMM Platforms: {platforms}")
if 'CUDA' in platforms:
    print(" -> GPU Acceleration Enabled (CUDA ready)!")
else:
    print(" -> Running on CPU. (Tip: Change runtime type to T4 GPU for faster execution)")

try:
    test_mol = Molecule.from_smiles("CC(=O)O")
    print(" -> OpenFF Toolkit is active and ready for automatic GAFF parameterization.")
except Exception as e:
    print(f" -> OpenFF Error: {e}")

# ==============================================================================
# STEP 1: BUILD REPEAT-UNIT POLYMER CHAINS (replaces monomer-only construction)
# ==============================================================================
print(f"\n[1/6] Building PLA (DP={DP_PLA}) and PCL (DP={DP_PCL}) polymer chains...")

# Repeat-unit SMILES fragments (see module docstring for the chemistry).
# Each fragment is written so that simple string repetition + a trailing "O"
# cap produces a valid, correctly-connected linear polyester chain.
PLA_REPEAT_UNIT = "O[C@@H](C)C(=O)"   # -[O-CH(CH3)-C(=O)]-, isotactic (PLLA)
PCL_REPEAT_UNIT = "OCCCCCC(=O)"       # -[O-(CH2)5-C(=O)]-


def build_linear_polyester_smiles(repeat_unit_smiles, degree_of_polymerization):
    """
    Assemble the SMILES string for a linear, unbranched polyester chain by
    repeating `repeat_unit_smiles` `degree_of_polymerization` times and
    capping the ends with a terminal hydroxyl (-OH, implicit on the first
    atom) and a terminal carboxylic acid (-COOH, via a trailing "O").

    This is valid ONLY for repeat units written head-to-tail as
    "O...C(=O)" (i.e. starting with the ester oxygen and ending with the
    carbonyl carbon), which is the convention used for PLA_REPEAT_UNIT and
    PCL_REPEAT_UNIT above. Do not reuse this function for branched or
    differently-oriented repeat units without checking the resulting SMILES.

    Parameters
    ----------
    repeat_unit_smiles : str
        SMILES fragment for one repeat unit, head-to-tail oriented.
    degree_of_polymerization : int
        Number of repeat units in the chain (DP >= 1).

    Returns
    -------
    str
        SMILES string for the full linear chain.
    """
    if degree_of_polymerization < 1:
        raise ValueError("degree_of_polymerization must be >= 1")
    return (repeat_unit_smiles * degree_of_polymerization) + "O"


def smiles_to_rdkit_polymer(smiles, n_embed_attempts=N_EMBED_ATTEMPTS,
                             random_seed=GLOBAL_RANDOM_SEED, label=""):
    """
    Converts a SMILES string to an RDKit 3D molecule with explicit
    hydrogens and defined stereochemistry, with retry logic for conformer
    embedding (longer polymer chains are more prone to embedding failure
    with a single attempt than small monomers were).

    Parameters
    ----------
    smiles : str
        SMILES string of the (poly)molecule to embed.
    n_embed_attempts : int
        Number of randomized embedding attempts (ETKDGv3) before giving up.
    random_seed : int
        Seed for reproducible conformer generation.
    label : str
        Human-readable name used only in diagnostic print statements.

    Returns
    -------
    rdkit.Chem.Mol
        3D-embedded, UFF-optimized molecule with explicit hydrogens.
    """
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        raise ValueError(f"[{label}] RDKit failed to parse SMILES:\n{smiles}")
    mol = Chem.AddHs(mol)

    params = AllChem.ETKDGv3()
    params.randomSeed = random_seed
    params.useRandomCoords = True
    params.maxIterations = 1000

    conf_id = -1
    for attempt in range(n_embed_attempts):
        params.randomSeed = random_seed + attempt
        conf_id = AllChem.EmbedMolecule(mol, params)
        if conf_id == 0:
            break
    if conf_id != 0:
        raise RuntimeError(
            f"[{label}] Conformer embedding failed after {n_embed_attempts} "
            f"attempts. For long chains, consider building the polymer "
            f"incrementally (embed a short seed chain, then extend and "
            f"re-optimize in stages) rather than embedding the full chain "
            f"from a random start."
        )

    ff_converged = AllChem.UFFOptimizeMolecule(mol, maxIters=2000)
    if ff_converged != 0:
        print(f" -> [WARNING] [{label}] UFF optimization did not fully "
              f"converge (status={ff_converged}); geometry may be strained.")

    Chem.AssignStereochemistry(mol, force=True, cleanIt=True)
    return mol


def report_polymer_stats(mol, label, degree_of_polymerization):
    """
    Prints a sanity-check summary (formula, molecular weight, atom count,
    expected repeat count) so the user can visually confirm the built
    molecule is a chain of the intended length, not a monomer or a
    malformed structure, before it is packed into a simulation box.
    """
    formula = Chem.rdMolDescriptors.CalcMolFormula(mol)
    mw = Descriptors.MolWt(mol)
    n_atoms_heavy = mol.GetNumAtoms() - sum(
        1 for atom in mol.GetAtoms() if atom.GetSymbol() == "H"
    )
    print(f" -> [{label}] DP={degree_of_polymerization} | Formula={formula} "
          f"| MW={mw:.1f} g/mol | Heavy atoms={n_atoms_heavy}")


print(" -> Constructing PLA chain SMILES...")
smiles_pla_chain = build_linear_polyester_smiles(PLA_REPEAT_UNIT, DP_PLA)
mol_pla = smiles_to_rdkit_polymer(smiles_pla_chain, label="PLA")
report_polymer_stats(mol_pla, "PLA", DP_PLA)

print(" -> Constructing PCL chain SMILES...")
smiles_pcl_chain = build_linear_polyester_smiles(PCL_REPEAT_UNIT, DP_PCL)
mol_pcl = smiles_to_rdkit_polymer(smiles_pcl_chain, label="PCL")
report_polymer_stats(mol_pcl, "PCL", DP_PCL)

# Export PDB for structural input / template generation, as in the original script
Chem.MolToPDBFile(mol_pla, "pla_chain.pdb")
files.download('pla_chain.pdb')
Chem.MolToPDBFile(mol_pcl, "pcl_chain.pdb")
files.download('pcl_chain.pdb')

# Convert RDKit objects -> OpenFF Molecule objects (with fallback flag)
openff_pla = Molecule.from_rdkit(mol_pla, allow_undefined_stereo=True)
openff_pcl = Molecule.from_rdkit(mol_pcl, allow_undefined_stereo=True)

# ocaya - edit: to switch partial charge method

# --- Switch charge method: assign charges up front so GAFFTemplateGenerator
# --- detects nonzero charges already present and skips AM1-BCC/antechamber.
print(" -> Assigning partial charges via Gasteiger (skipping AM1-BCC for speed)...")
openff_pla.assign_partial_charges(partial_charge_method="gasteiger")
openff_pcl.assign_partial_charges(partial_charge_method="gasteiger")

# ocaya - end edit

print("[SUCCESS] OpenFF polymer-chain Molecules generated with assigned stereochemistry.")

# --- PACKMOL PACKING (unchanged in structure from the original script) ---
print("\n[1.5/6] Packing Multiple PLA and PCL Chains...")

selected_type = globals().pop('_GLOBAL_OVERRIDE_POLYMER_TYPE_', None)

if selected_type is None:
    n_pla_chains_current = getattr(globals().get('s_n_pla_chains'), 'value', 50)
    n_pcl_chains_current = getattr(globals().get('s_n_pcl_chains'), 'value', 50)
    selected_type = getattr(globals().get('s_polymer_type'), 'value', 'Blend')
else:
    n_pla_chains_current = getattr(globals().get('s_n_pla_chains'), 'value', 50)
    n_pcl_chains_current = getattr(globals().get('s_n_pcl_chains'), 'value', 50)

molecules_to_pack_unique = []
number_of_copies = []

if selected_type == 'Pure PLA':
    n_pcl_chains_current = 0
    molecules_to_pack_unique.append(openff_pla)
    number_of_copies.append(n_pla_chains_current)
    print(f"Selected: Pure PLA ({n_pla_chains_current} chains, DP={DP_PLA})")
elif selected_type == 'Pure PCL':
    n_pla_chains_current = 0
    molecules_to_pack_unique.append(openff_pcl)
    number_of_copies.append(n_pcl_chains_current)
    print(f"Selected: Pure PCL ({n_pcl_chains_current} chains, DP={DP_PCL})")
else:  # 'Blend'
    molecules_to_pack_unique.extend([openff_pla, openff_pcl])
    number_of_copies.extend([n_pla_chains_current, n_pcl_chains_current])
    print(f"Selected: Blend (PLA: {n_pla_chains_current} chains @ DP={DP_PLA}, "
          f"PCL: {n_pcl_chains_current} chains @ DP={DP_PCL})")

# Polymer chains are much larger than the earlier monomers, so a larger box
# is generally needed to pack the same number of chains without excessive
# initial overlap/strain. Adjust packing_box_size upward if pack_box fails
# to converge or minimization energies are pathologically high afterwards.
# Doubling the box size from 6.0 nm to 12.0 nm
packing_box_size = 12.0 * unit.nanometers

packed_system_pdb_file = "packed_system.pdb"
box_size_nm = packing_box_size.value_in_unit(unit.nanometer)
box_vectors_array = np.array([
    [box_size_nm, 0.0, 0.0],
    [0.0, box_size_nm, 0.0],
    [0.0, 0.0, box_size_nm]
])
box_vectors_packmol = openff_unit.Quantity(box_vectors_array, openff_unit.nanometer)

# Create a persistent working directory for Packmol
packmol_working_directory = '/content/packmol_temp'
os.makedirs(packmol_working_directory, exist_ok=True)

if not molecules_to_pack_unique:
    raise ValueError("No molecules selected for packing. Please select 'Pure PLA', 'Pure PCL', or 'Blend' using the dropdown.")

packed_system = pack_box(
    molecules=molecules_to_pack_unique,
    number_of_copies=number_of_copies,
    box_vectors=box_vectors_packmol,
    tolerance=2.0 * openff_unit.angstrom,
    working_directory=packmol_working_directory # Pass the persistent working directory
)

packed_system.to_file(packed_system_pdb_file, file_format="pdb")
files.download('packed_system.pdb')

print(f"[SUCCESS] Packed {n_pla_chains_current} PLA chains (DP={DP_PLA}) and "
      f"{n_pcl_chains_current} PCL chains (DP={DP_PCL}) into {packed_system_pdb_file}")

# ==============================================================================
# STEP 2: BUILD PERIODIC BOX & APPLY GAFF FORCE FIELD
# ==============================================================================
print("\n[2/6] Constructing Simulation Box and Assigning GAFF Parameters...")

pdb_packed = app.PDBFile(packed_system_pdb_file)

forcefield = app.ForceField('amber14-all.xml', 'amber14/tip3p.xml')

gaff_molecules_to_register = []
if openff_pla in molecules_to_pack_unique:
    gaff_molecules_to_register.append(openff_pla)
if openff_pcl in molecules_to_pack_unique:
    gaff_molecules_to_register.append(openff_pcl)

if not gaff_molecules_to_register:
    raise ValueError("No molecules selected for GAFF parameterization. Please select a polymer type.")

gaff = GAFFTemplateGenerator(molecules=gaff_molecules_to_register, forcefield='gaff-2.11')
forcefield.registerTemplateGenerator(gaff.generator)

system = forcefield.createSystem(
    pdb_packed.topology,
    nonbondedMethod=app.PME,
    nonbondedCutoff=1.0 * unit.nanometers,
    constraints=app.HBonds
)

# ==============================================================================
# STEP 3: INTEGRATOR & BAROSTAT SETUP
# ==============================================================================
print("\n[3/6] Setting up Langevin Integrator and Monte Carlo Barostat...")

initial_temp = 450.0 * unit.kelvin
pressure = 1.0 * unit.atmospheres
timestep = 2.0 * unit.femtoseconds

integrator = mm.LangevinMiddleIntegrator(
    initial_temp,
    1.0 / unit.picosecond,
    timestep
)
integrator.setRandomNumberSeed(GLOBAL_RANDOM_SEED)

barostat = mm.MonteCarloBarostat(pressure, initial_temp, 25)
barostat.setRandomNumberSeed(GLOBAL_RANDOM_SEED)
system.addForce(barostat)

try:
    platform = mm.Platform.getPlatformByName('CUDA')
    properties = {'Precision': 'mixed'}
    print(" -> Utilizing NVIDIA GPU Acceleration (CUDA).")
except Exception:
    platform = mm.Platform.getPlatformByName('CPU')
    properties = {}
    print(" -> CUDA unavailable. Falling back to CPU Execution.")

simulation = app.Simulation(
    pdb_packed.topology,
    system,
    integrator,
    platform,
    properties
)
simulation.context.setPositions(pdb_packed.positions)

# ==============================================================================
# STEP 4: ENERGY MINIMIZATION & NVT MELT RELAXATION
# ==============================================================================
print("\n[4/6] Executing L-BFGS Energy Minimization...")
simulation.minimizeEnergy(maxIterations=1000)

print(" -> Running High-T NVT Melt Relaxation at 450 K...")
simulation.context.setVelocitiesToTemperature(initial_temp, GLOBAL_RANDOM_SEED)
simulation.step(50000)  # 100 ps

# ==============================================================================
# STEP 5: NPT STEP-WISE THERMAL ANNEALING (450 K -> 250 K)
# ==============================================================================
print("\n[5/6] Starting NPT Thermal Annealing Loop for Tg Extraction...")
print(" -> NOTE: see 'KNOWN LIMITATIONS' item 4 at the top of this file if "
      "running a Pure PCL case -- the range below may not reach PCL's real Tg.")

temps_to_simulate = np.arange(450, 230, -20)  # 450K, 430K, ..., 250K
annealing_results = []

for T in temps_to_simulate:
    target_temp = T * unit.kelvin
    integrator.setTemperature(target_temp)
    barostat.setDefaultTemperature(target_temp)

    simulation.step(20000)

    state = simulation.context.getState(getPositions=True, getEnergy=True)
    box_vectors = state.getPeriodicBoxVectors(asNumpy=True)
    volume = box_vectors[0][0] * box_vectors[1][1] * box_vectors[2][2]

    total_mass = sum([atom.element.mass.value_in_unit(unit.dalton)
                      for atom in pdb_packed.topology.atoms()])

    vol_cm3 = volume.value_in_unit(unit.centimeter**3)
    mass_g = total_mass * 1.66053906660e-24
    density_val = (mass_g / vol_cm3)

    annealing_results.append((T, density_val))
    print(f" -> T = {T:3d} K | Density = {density_val:.4f} g/cm3")

# ==============================================================================
# STEP 6: PLOT DENSITY vs TEMPERATURE AND EXTRACT Tg
# ==============================================================================
temps, densities = zip(*annealing_results)

n_pla = n_pla_chains_current
n_pcl = n_pcl_chains_current

base_title = "Mass Density vs. Temperature (NPT Cooling)"
if selected_type == 'Pure PLA':
    dynamic_part = f"(Pure PLA, {n_pla} chains, DP={DP_PLA})"
elif selected_type == 'Pure PCL':
    dynamic_part = f"(Pure PCL, {n_pcl} chains, DP={DP_PCL})"
else:  # 'Blend'
    dynamic_part = f"(Blend: PLA {n_pla} chains @ DP={DP_PLA}, PCL {n_pcl} chains @ DP={DP_PCL})"

plot_title = f"{base_title} {dynamic_part}"

# Plotting
plt.figure(figsize=(10, 6))
plt.plot(temps, densities, 'o-', markersize=8)
plt.title(plot_title)
plt.xlabel('Temperature (K)')
plt.ylabel('Density (g/cm$^3$)')
plt.grid(True)

# Linear fit for the two regions (simplified for example)
# This is a basic example; more robust Tg extraction often involves more data points and fitting methods
# Here, we'll just fit a single line to show the principle.

# For a more accurate Tg, you'd fit two lines to the glassy and rubbery regions
# and find their intersection. This often requires more data points, especially
# around the expected Tg, and careful selection of the fitting ranges.

# Let's assume we want to fit the entire range for demonstration purposes if a simple intersection is not immediately visible
if len(temps) > 1:
    # Use curve_fit to fit a straight line to the data
    def linear_function(x, a, b):
        return a * x + b

    try:
        # Fit to all points for a simplified view
        popt, pcov = curve_fit(linear_function, temps, densities)
        plt.plot(temps, linear_function(np.array(temps), *popt), 'r--', label='Linear Fit')
        print(f"\nLinear fit: Density = {popt[0]:.4e} * T + {popt[1]:.4f}")
    except RuntimeError:
        print("\nCould not perform linear fit. Not enough data points or poor convergence.")

plt.legend()
plt.show()

print("\n[SUCCESS] Density vs. Temperature plot generated.")

✨🍰✨ Everything looks OK!
conda-forge/linux-64                                        Using cache
conda-forge/noarch                                          Using cache
[+] 0.0s
[+] 0.0s

Pinned packages:

  - python=3.12

Pinned packages:

  - python_abi[version="=3.12",build="*cp312*"]

Pinned packages:

  - cuda-version*.*

Pinned packages:

  - python=3.12


Transaction

  Prefix: /usr/local

  All requested packages already installed


Transaction starting
[+] 0.0s

Transaction finished

conda-forge/linux-64                                        Using cache
conda-forge/noarch                                          Using cache
[+] 0.0s
[+] 0.0s

Pinned packages:

  - python=3.12

Pinned packages:

  - python_abi[version="=3.12",build="*cp312*"]

Pinned packages:

  - cuda-version*.*

Pinned packages:

  - python=3.12


Transaction

  Prefix: /usr/local

  All requested packages already installed


Transaction starting
[+] 0.0s

Transaction finished

Verifying openff-packmol inst

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

 -> Assigning partial charges via Gasteiger (skipping AM1-BCC for speed)...
[SUCCESS] OpenFF polymer-chain Molecules generated with assigned stereochemistry.

[1.5/6] Packing Multiple PLA and PCL Chains...
Selected: Blend (PLA: 50 chains @ DP=20, PCL: 50 chains @ DP=20)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

[SUCCESS] Packed 50 PLA chains (DP=20) and 50 PCL chains (DP=20) into packed_system.pdb

[2/6] Constructing Simulation Box and Assigning GAFF Parameters...

[3/6] Setting up Langevin Integrator and Monte Carlo Barostat...
 -> CUDA unavailable. Falling back to CPU Execution.

[4/6] Executing L-BFGS Energy Minimization...
 -> Running High-T NVT Melt Relaxation at 450 K...

[5/6] Starting NPT Thermal Annealing Loop for Tg Extraction...
 -> NOTE: see 'KNOWN LIMITATIONS' item 4 at the top of this file if running a Pure PCL case -- the range below may not reach PCL's real Tg.
 -> T = 450 K | Density = 0.9358 g/cm3
 -> T = 430 K | Density = 0.9895 g/cm3
 -> T = 410 K | Density = 1.0054 g/cm3
 -> T = 390 K | Density = 1.0217 g/cm3
 -> T = 370 K | Density = 1.0334 g/cm3
 -> T = 350 K | Density = 1.0520 g/cm3
 -> T = 330 K | Density = 1.0643 g/cm3
 -> T = 310 K | Density = 1.0718 g/cm3
